In [1]:
import pandas as pd
import numpy as np

In [2]:
df_customers= pd.read_csv("/content/AWCustomers.csv")
df_sales= pd.read_csv("/content/AWSales.csv")

# Merge customers and sales on CustomerID
df_merged = pd.merge(df_customers, df_sales, on="CustomerID")

# Create Age from BirthDate
df_merged["BirthDate"] = pd.to_datetime(df_merged["BirthDate"], errors="coerce")
df_merged["Age"] = pd.Timestamp.now().year - df_merged["BirthDate"].dt.year

In [3]:
df_merged.head()

,CustomerID,Title,FirstName,MiddleName,LastName,Suffix,AddressLine1,AddressLine2,City,StateProvinceName,...,MaritalStatus,HomeOwnerFlag,NumberCarsOwned,NumberChildrenAtHome,TotalChildren,YearlyIncome,LastUpdated,BikeBuyer,AvgMonthSpend,Age
0,21173,NaN,Chad,C,Yuan,NaN,7090 C. Mount Hood,NaN,Wollongong,New South Wales,...,M,1,3,0,1,81916,2017-03-06,1,50.97,38
1,13249,NaN,Ryan,NaN,Perry,NaN,3651 Willow Lake Rd,NaN,Shawnee,British Columbia,...,M,1,2,1,2,81076,2017-03-06,1,53.11,53
2,29350,NaN,Julia,NaN,Thompson,NaN,1774 Tice Valley Blvd.,NaN,West Covina,California,...,S,0,3,0,0,86387,2017-03-06,1,54.08,40
3,13503,NaN,Theodore,NaN,Gomez,NaN,2103 Baldwin Dr,NaN,Liverpool,England,...,M,1,2,1,2,61481,2017-03-06,1,56.93,48
4,22803,NaN,Marshall,J,Shan,NaN,Am Gallberg 234,NaN,Werne,Nordrhein-Westfalen,...,S,1,1,0,0,51804,2017-03-06,1,55.41,50


In [4]:
selected_features = [
    "Age", "Gender", "MaritalStatus", "Education", "Occupation",
    "HomeOwnerFlag", "NumberCarsOwned", "NumberChildrenAtHome",
    "TotalChildren", "YearlyIncome", "AvgMonthSpend", "BikeBuyer"
]
df_selected = df_merged[selected_features].copy()

In [5]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, KBinsDiscretizer, OneHotEncoder
import numpy as np

# (a) Handle missing values
df_selected = df_selected.dropna()

# (b) Normalize Age, Income, Spend
scaler = MinMaxScaler()
df_selected[["Age","YearlyIncome","AvgMonthSpend"]] = scaler.fit_transform(
    df_selected[["Age","YearlyIncome","AvgMonthSpend"]])

# (c) Discretize Age into 5 bins
binning = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
df_selected["Age_binned"] = binning.fit_transform(df_selected[["Age"]])

# (d) Standardize YearlyIncome
standardizer = StandardScaler()
df_selected[["YearlyIncome"]] = standardizer.fit_transform(df_selected[["YearlyIncome"]])


In [6]:
# (e) One Hot Encoding for categorical vars
encoder = OneHotEncoder(drop='first', sparse_output=False)
encoded = encoder.fit_transform(df_selected[["Gender","MaritalStatus","Education","Occupation"]])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out())

# Final transformed dataset
df_final = pd.concat([df_selected.drop(["Gender","MaritalStatus","Education","Occupation"], axis=1).reset_index(drop=True),
                      encoded_df.reset_index(drop=True)], axis=1)

df_final.head()

,Age,HomeOwnerFlag,NumberCarsOwned,NumberChildrenAtHome,TotalChildren,YearlyIncome,AvgMonthSpend,BikeBuyer,Age_binned,Gender_M,MaritalStatus_S,Education_Graduate Degree,Education_High School,Education_Partial College,Education_Partial High School,Occupation_Management,Occupation_Manual,Occupation_Professional,Occupation_Skilled Manual
0,0.185714,1,3,0,1,0.298555,0.324210,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.400000,1,2,1,2,0.271180,0.425201,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.214286,0,3,0,0,0.444261,0.470977,1,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.328571,1,2,1,2,-0.367401,0.605474,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,0.357143,1,1,0,0,-0.682765,0.533742,1,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [7]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import jaccard

# Select two records
obj1 = df_final.iloc[0].values.reshape(1, -1)
obj2 = df_final.iloc[1].values.reshape(1, -1)

# Cosine Similarity
cos_sim = cosine_similarity(obj1, obj2)[0][0]

# Jaccard Similarity (binary/categorical attributes)
jacc_sim = 1 - jaccard(obj1[0], obj2[0])

# Simple Matching Coefficient
smc = np.mean(obj1[0] == obj2[0])

print("Cosine Similarity:", cos_sim)
print("Jaccard Similarity:", jacc_sim)
print("Simple Matching Coefficient:", smc)

# Correlation between YearlyIncome and AvgMonthSpend
corr = df_final["YearlyIncome"].corr(df_final["AvgMonthSpend"])
print("Correlation between YearlyIncome & AvgMonthSpend:", corr)

Cosine Similarity: 0.8178182000976661
Jaccard Similarity: 0.7272727272727273
Simple Matching Coefficient: 0.5789473684210527
Correlation between YearlyIncome & AvgMonthSpend: 0.5301257155563447
